# PCB 결함 검????YOLOv8 ?�이?�파?��????�닝 Colab ?�트�?n
**Team Convex | ATI ?�턴 ?�로?�트**

???�트북�? `scripts/run_tune.bat`�?**?�일???�닝 ?�름**??Google Colab?�서 ?�행?�며,
Google Drive ?�동?�로 **?�션???�겨??결과가 Drive???�전?�게 보존**?�니??

> **?�전 ?�고리즘(GA) 기반 Ultralytics ?�장 ?�너�??�용?�니??**  
> ?�닝?� ?�반 ?�습보다 ?�씬 ?�랜 ?�간???�요?�니??(`iterations × epochs` ?�위).

---

## ?�행 ??준�?n
1. **GitHub Token ?�록** (Private Repo ?�근??
   - Colab ?�쪽 ?�이?�바 ?�� ?�이�??�릭
   - `GITHUB_TOKEN` ?�름?�로 Personal Access Token(PAT) 추�?
   - PAT ?�성: GitHub ??Settings ??Developer settings ??Personal access tokens
   - ?�요 권한: `Contents: Read`

2. **dataset.zip Drive ?�로??*
   - Google Drive `MyDrive/pcb-defect-inspection/` ?�더??`dataset.zip` ?�로??n
3. **?��????�형 ?�인**
   - 메뉴: ?��??????��????�형 변�???**T4 GPU** ?�택

---

## ?�습 ?�름 (run_tune.bat�??�일)

| ?� | ?�계 | 비고 |
|----|------|------|
| ??| ?�경 ?�정 (GPU ?�인) | 경로 ?�수 ?�의 |
| ??| Google Drive 마운??| |
| ??| Repo ?�론 & ?�키지 ?�치 | Private repo 지??|
| ??| ?�정 ?�인 | tune ?�라미터 출력 |
| ??| ?�처�?| ?��? ?�료 ???�동 ?�킵 |
| ??| ?�전 ?�닝 결과 체크 | Drive??runs/tune ?�동 감�? |
| ??| ?�닝 ?�행 | src/tune.py ?�행 |
| ??| Drive ?�기??| ?�션 종료 ???�수! |
| ??| 결과 ?�인 | best_hyperparameters.yaml 출력 (?�택 ?�항) |

## ???�경 ?�정

> `DRIVE_PROJECT_DIR`???�제 Drive ?�더?� ?�르�??�정 ???�행?�세??

In [ ]:
# ============================================================
# ???�경 ?�정 ???�래 ?�수�??�인 ???�행?�세??n# ============================================================

# --- GitHub ?�정 ---
GITHUB_REPO   = "june0922/pcb-defect-inspection"
GITHUB_BRANCH = "main"  # ?�용??브랜치명
PROJECT_DIR   = "/content/pcb-defect-inspection"

# --- Google Drive 경로 ---
# dataset.zip???�로?�한 Drive ?�더 경로 (?�요???�정)
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/pcb-defect-inspection"
DRIVE_DATASET_ZIP = f"{DRIVE_PROJECT_DIR}/dataset.zip"

# --- GPU ?�인 ---
import subprocess
try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"??GPU: {r.stdout.strip()}")
    else:
        print("?�️  GPU ?�음 ???��????�형??'GPU'�?변경하?�요.")
        print("   메뉴: ?��???> ?��????�형 변�?> T4 GPU ?�택")
except FileNotFoundError:
    print("?�️  GPU ?�음 (nvidia-smi 명령?��? 찾을 ???�음) ???��????�형??'GPU'�?변경하?�요.")
    print("   메뉴: ?��???> ?��????�형 변�?> T4 GPU ?�택")

print(f"\n?�로?�트 ?�렉?�리 : {PROJECT_DIR}")
print(f"Drive ?�로?�트 ?�더: {DRIVE_PROJECT_DIR}")
print(f"Drive dataset.zip  : {DRIVE_DATASET_ZIP}")

## ??Google Drive 마운??


In [ ]:
# ============================================================
# ??Google Drive 마운??n# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print("??Drive 마운???�료")
print(f"Drive ?�로?�트 ?�더: {DRIVE_PROJECT_DIR}")
contents = os.listdir(DRIVE_PROJECT_DIR)
print(f"?�재 ?�용: {contents if contents else '(비어 ?�음)'}")

## ??Repo ?�론 & ?�키지 ?�치 (Private Repo)

> **?�전 준�?*: Colab ?�쪽 ?�� ?�이�???`GITHUB_TOKEN` ?�록 ?�요

In [ ]:
# ============================================================
# ??Repo ?�론 & ?�키지 ?�치 (Private Repo)
#    보안: subprocess�??�론???�큰??출력???�출?��? ?�습?�다.
# ============================================================
from google.colab import userdata
import subprocess, os

# --- GitHub Token 로드 ---
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("??GitHub Token 로드??(Colab Secrets)")
except Exception:
    GITHUB_TOKEN = ""
    print("??GITHUB_TOKEN??찾을 ???�습?�다.")
    print("   ?�결: ?�쪽 ?�이?�바 ??�??�릭 ??'GITHUB_TOKEN' 추�?")
    raise RuntimeError("GITHUB_TOKEN???�요?�니??")

# --- Repo ?�론 ?�는 ?�데?�트 ---
repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"

if not os.path.exists(PROJECT_DIR):
    print(f"'{GITHUB_REPO}' ?�론 �?.. (?�정 ?�더 ?�외 ?�정 ?�용)")
    result = subprocess.run(
        ["git", "clone", "--no-checkout", "--branch", GITHUB_BRANCH, repo_url, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        err = result.stderr.replace(GITHUB_TOKEN, "***")
        raise RuntimeError(f"?�론 ?�패:\n{err}")
    
    subprocess.run(["git", "-C", PROJECT_DIR, "config", "core.sparseCheckout", "true"], check=True)
    sparse_file = os.path.join(PROJECT_DIR, ".git", "info", "sparse-checkout")
    with open(sparse_file, "w", encoding="utf-8") as f:
        f.write("/*\n")
        f.write("!/.agents/\n")
        f.write("!/app/\n")
        f.write("!/runs/\n")
        f.write("!/web_*/\n")
        f.write("!/weights/\n")
    
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", GITHUB_BRANCH], capture_output=True, text=True, check=True)
    print(f"??Repo ?�론 ?�료: {PROJECT_DIR}")
else:
    print(f"?�️  ?��? ?�론??({PROJECT_DIR}) ??최신?�로 ?�데?�트?�니??")
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "remote", "set-url", "origin", repo_url],
        capture_output=True
    )
    result = subprocess.run(
        ["git", "-C", PROJECT_DIR, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or "?��? 최신 ?�태?�니??")

# --- ?�업 ?�렉?�리 ?�동 ---
%cd {PROJECT_DIR}
print(f"\n?�업 ?�렉?�리: {os.getcwd()}")

# --- ?�키지 ?�치 ---
print("\n?�키지 ?�치 �?..")
!pip install -q -r requirements.txt
print("???�키지 ?�치 ?�료")

## ???�정 ?�인

> ?�래 ?��? ?�인?�고 ?�정??맞으�??�음 ?��?진행?�세??  
> 변경이 ?�요?�면 `config.yaml`??직접 ?�정 ?????�???�시 ?�행?�세??  
> **Tune 관???�라미터**: `tune.iterations` (기본 30??, `tune.epochs` (반복???�포?? 기본 30)

In [ ]:
# ============================================================
# ??config.yaml ??env=colab ?�치 �??�정 출력
#    (run_tune.bat??show_config.py + env ?�환 ??��)
# ============================================================
import yaml, sys, os
sys.path.insert(0, "src")

with open("config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# env�?colab?�로 ?�환
cfg["env"] = "colab"
with open("config.yaml", "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, allow_unicode=True)

# ?�정 출력 (show_config.py?� ?�일???�이�??�식)
def print_dict(d, prefix=''):
    for k, v in d.items():
        if isinstance(v, dict):
            print_dict(v, prefix + k + '.')
        else:
            v_str = ', '.join(map(str, v)) if isinstance(v, list) else str(v)
            print(f"| {prefix+k:<33} | {v_str:<25} |")

print("+" + "-"*35 + "+" + "-"*27 + "+")
print(f"| {'Parameter':<33} | {'Value':<25} |")
print("+" + "-"*35 + "+" + "-"*27 + "+")
print_dict(cfg)
print("+" + "-"*35 + "+" + "-"*27 + "+")
print()

# Tune ?�라미터 별도 강조 출력
tc = cfg.get("tune", {})
iterations = tc.get('iterations', 100)
epochs     = tc.get('epochs', 15)
total_est  = iterations * epochs
print(f"?�� Tune ?�정: iterations={iterations}??× epochs={epochs} = �?~{total_est} ?�포??")
print(f"   모델: {tc.get('model', 'weights/yolov8n.pt')} | imgsz: {tc.get('imgsz', 640)}")
print()
print("?�️  ???�정???�인 ?? 문제?�으�??�음 ?�(???�처�????�행?�세??")

## ???�처�?n
- `preprocessed_data/` ?�더가 ?��? ?�으�?**?�동?�로 건너?�니??*
- 강제 ?�처�? `FORCE_PREPROCESS = True` �?변�????�행

> **?�전 준�?*: Drive `pcb-defect-inspection/dataset.zip` ?�로???�요

In [ ]:
# ============================================================
# ???�이???�처�?(run_tune.bat??preprocess ?�택 ?�계)
#    preprocessed_data/ 가 ?�으�??�킵, ?�으�?Drive?�서 zip 복사 ???�행
# ============================================================
import sys, os, shutil
from pathlib import Path
sys.path.insert(0, "src")
from utils import load_config, get_paths

FORCE_PREPROCESS = False  # True�?변경하�?강제 ?�처�?n
cfg = load_config("config.yaml")
paths = get_paths(cfg)
processed_dir = paths["processed"]
train_img_dir = processed_dir / "images" / "train"

already_done = train_img_dir.exists() and any(train_img_dir.glob("*.jpg"))

if already_done and not FORCE_PREPROCESS:
    n_train = len(list((processed_dir / "images" / "train").glob("*.jpg")))
    n_val   = len(list((processed_dir / "images" / "val").glob("*.jpg")))
    n_test  = len(list((processed_dir / "images" / "test").glob("*.jpg")))
    print("?�️  ?��? ?�처리된 ?�이?��? ?�습?�다 ??건너?�니??")
    print(f"   train: {n_train}??/ val: {n_val}??/ test: {n_test}??")
    print("   강제 ?�처�? FORCE_PREPROCESS = True �?변�????�실??")
else:
    # --- dataset.zip 준�?---
    local_zip = Path(PROJECT_DIR) / "dataset.zip"
    if not local_zip.exists():
        if os.path.exists(DRIVE_DATASET_ZIP):
            size_mb = os.path.getsize(DRIVE_DATASET_ZIP) / 1e6
            print(f"Drive?�서 dataset.zip 복사 �?.. ({size_mb:.1f} MB)")
            shutil.copy2(DRIVE_DATASET_ZIP, local_zip)
            print("??복사 ?�료")
        else:
            msg = (
                "dataset.zip??찾을 ???�습?�다.\n"
                f"Drive 경로: {DRIVE_DATASET_ZIP}\n"
                f"Local 경로: {local_zip}\n"
                "Google Drive??pcb-defect-inspection/ ?�더??dataset.zip???�로?�하?�요."
            )
            raise FileNotFoundError(msg)
    else:
        size_mb = local_zip.stat().st_size / 1e6
        print(f"?�️  dataset.zip 로컬??존재 ({size_mb:.1f} MB)")

    # --- ?�처�??�행 ---
    print("\n?�처�??�작...")
    !python src/preprocess.py --config config.yaml
    print("\n???�처�??�료")

## ???�전 ?�닝 결과 체크

Drive??`runs/tune` ?�더�??�동?�로 감�??�고, Drive?� ?�시�??�기???�결(?�볼�?링크)???�정?�니??

- **`runs/tune` ?�음** ?????�닝 ?�작
- **`runs/tune` ?�음** ??기존 결과�???��?��? ?��?�??�택 (`OVERWRITE_TUNE`)

| 변??| ?�작 |
|------|------|
| `OVERWRITE_TUNE = None` | **?�동 감�?** (기본�? 권장) |
| `OVERWRITE_TUNE = True` | ?�전 결과 무시?�고 ???�닝 ?�작 |
| `OVERWRITE_TUNE = False` | ?�전 결과 ?�으�??�닝 취소 |

> **주의**: YOLO tune?� `exist_ok=True`�??�행?��?�??�전 결과가 ?�어?????�닝???�어??진행?�니??

In [ ]:
# ============================================================
# ???�전 ?�닝 결과 체크 & Drive ?�시�??�기???�정
#    runs ?�더�?Google Drive???�볼�?링크�??�결?�여
#    ?�닝 �??�션???�겨??결과가 Drive???�시�??�?�되?�록 ??
# ============================================================
import os, shutil
from pathlib import Path

# None=?�동 감�?, True=?�전 결과 무시 ???�닝, False=?�전 결과 ?�으�?취소
OVERWRITE_TUNE = None

DRIVE_RUNS = f"{DRIVE_PROJECT_DIR}/runs"
LOCAL_RUNS = f"{PROJECT_DIR}/runs"

# 1. Drive ?�시�??�?�을 ?�한 ?�볼�?링크(Symlink) ?�정
os.makedirs(DRIVE_RUNS, exist_ok=True)
if os.path.exists(LOCAL_RUNS) and not os.path.islink(LOCAL_RUNS):
    shutil.rmtree(LOCAL_RUNS)  # 기존 로컬 ?�더 ?�거
if not os.path.exists(LOCAL_RUNS):
    os.symlink(DRIVE_RUNS, LOCAL_RUNS)  # 로컬 runs -> Drive runs ?�결
print("[Drive ?�기?? 로컬 runs/ ?�더가 Google Drive???�시�??�결?�었?�니??")

# 2. ?�전 ?�닝 결과 ?�인
DRIVE_TUNE_DIR = Path(DRIVE_RUNS) / "tune"
tune_exists = DRIVE_TUNE_DIR.exists() and any(DRIVE_TUNE_DIR.iterdir())

print()
if tune_exists:
    best_hp = DRIVE_TUNE_DIR / "best_hyperparameters.yaml"
    print(f"?�️  ?�전 ?�닝 결과 발견: {DRIVE_TUNE_DIR}")
    if best_hp.exists():
        print(f"   best_hyperparameters.yaml ??존재")
    else:
        print(f"   best_hyperparameters.yaml ??미생??(?�닝 중단??것으�?보임)")
else:
    print("?�️  ?�전 ?�닝 결과 ?�음 ?????�닝???�작?�니??")

# 3. ?�행 ?��? 결정
if OVERWRITE_TUNE is None:
    PROCEED_TUNE = True  # ?�동 감�?: tune.py ?��??�서 resume ?��? 결정
    if tune_exists:
        tune_ndjson = DRIVE_TUNE_DIR / "tune_results.ndjson"
        if tune_ndjson.exists():
            completed = sum(1 for l in tune_ndjson.open(encoding='utf-8') if l.strip())
            print(f"   ???�전 {completed}�?iteration 감�? ??src/tune.py가 resume=True�??�어??진행?�니??")
        else:
            print("   ??tune ?�더???�으??tune_results.ndjson ?�음 ??처음부???�작?�니??")
        print("   ???�전???�로 ?�작?�려�? OVERWRITE_TUNE = True�?변�????�실??")
elif OVERWRITE_TUNE:
    # ?�닝 ?�렉?�리�???��?�야 tune.py가 resume=False�?�??�작??n    if DRIVE_TUNE_DIR.exists():
        shutil.rmtree(DRIVE_TUNE_DIR)
        print("???�전 ?�닝 결과 ??�� ?�료")
    PROCEED_TUNE = True
    print("OVERWRITE_TUNE = True ???�전 결과 ??�� ?????�닝 ?�작")
else:  # OVERWRITE_TUNE = False
    if tune_exists:
        PROCEED_TUNE = False
        print("OVERWRITE_TUNE = False ???�전 결과가 ?�어 ?�닝??취소?�니??")
    else:
        PROCEED_TUNE = True

print()
print("=" * 42)
print(f"  PROCEED_TUNE = {PROCEED_TUNE}")
if PROCEED_TUNE:
    print("  ?? ?�음 ?�: ?�이?�파?��????�닝 ?�작")
else:
    print("  ?�️  ?�닝??취소?�었?�니??")
    print("  ?�행?�려�? OVERWRITE_TUNE = True�?변�????�실??")
print("=" * 42)

## ???�닝 ?�행

> ???�??`PROCEED_TUNE` 값이 `True`???�만 ?�행?�니??  
> ?�닝 결과??`runs/tune/` ?�더???�?�되�? Drive?� ?�시�??�기?�됩?�다.  
>
> **?�??경로**:
> - ?�닝 결과 ?�체: `runs/tune/`
> - 최적 ?�이?�파?��??? `runs/tune/best_hyperparameters.yaml`

In [ ]:
# ============================================================
# ???�이?�파?��????�닝 ?�행 (run_tune.bat??python src/tune.py ??��)
# ============================================================
if not PROCEED_TUNE:
    print("?�️  PROCEED_TUNE = False ???�닝??취소?�었?�니??")
    print("   ???�?�서 OVERWRITE_TUNE = True�?변�????�시 ?�행?�세??")
else:
    import yaml
    # Tune ?�라미터 ?�약 출력
    with open("config.yaml", "r", encoding="utf-8") as f:
        _cfg = yaml.safe_load(f)
    tc = _cfg.get("tune", {})
    iterations = tc.get('iterations', 100)
    epochs     = tc.get('epochs', 15)

    print("=" * 52)
    print("  ?? ?�이?�파?��????�닝 ?�작")
    print(f"  iterations: {iterations}??| epochs/iter: {epochs}")
    print(f"  �??�상 ?�포?? ~{iterations * epochs}")
    print("  결과: runs/tune/best_hyperparameters.yaml")
    print("=" * 52)
    print()

    !python src/tune.py --config config.yaml

    print()
    print("=" * 52)
    print("  ?�닝 종료 (?�는 ?�션 중단)")
    print("  ??반드???�음 ?� ??Drive ?�기?��? ?�행?�세??")
    print("=" * 52)

## ??Drive ?�기??(?�택/?�중?�전)

> ???�?�서 ?�볼�?링크�??�정?�다�????�???�어??**?�닝 결과??Drive???�전**?�니??  
> 
> ?�만 최종?�으�?`weights/` ?�더�?복사?�두�??�해 ?�행?�두�?좋습?�다.

In [ ]:
# ============================================================
# ??Drive ?�기??(?�중?�전)
# ============================================================
import shutil, os
from pathlib import Path

def sync_dir_to_drive(local_dir: str, drive_dir: str) -> int:
    local_path = Path(local_dir)
    drive_path = Path(drive_dir)
    if not local_path.exists():
        print(f"  ?�️  ?�음 (건너?�): {local_dir}")
        return 0

    # If local_dir is a symlink to drive_dir, skip copying files within it
    # as they are already synchronized.
    if local_path.is_symlink() and local_path.resolve() == drive_path.resolve():
        print(f"  ?�️  '{local_dir}'???��? Drive???�볼�?링크?�어 ?�습?�다 ??복사 건너?�.")
        return 0

    drive_path.mkdir(parents=True, exist_ok=True)
    count = 0
    for src in local_path.rglob("*"):
        if src.is_file():
            dst = drive_path / src.relative_to(local_path)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            count += 1
    return count

print("Drive ?�기??�?..\n")

# runs/ ?�기??(runs/tune/ ?�더 ?�함)
n1 = sync_dir_to_drive(
    f"{PROJECT_DIR}/runs",
    f"{DRIVE_PROJECT_DIR}/runs"
)
print(f"  ??runs/    ??Drive ({n1}�??�일)")

# weights/ ?�기??nn2 = sync_dir_to_drive(
    f"{PROJECT_DIR}/weights",
    f"{DRIVE_PROJECT_DIR}/weights"
)
print(f"  ??weights/ ??Drive ({n2}�??�일)")

print()
print(f"?�� ?�기???�료 ??{DRIVE_PROJECT_DIR}")
print("   최적 ?�이?�파?��??? runs/tune/best_hyperparameters.yaml")

## ??결과 ?�인 (?�택 ?�항)

?�닝???�료????최적 ?�이?�파?��??��? ?�닝 결과 그래?��? ?�인?�니??

In [ ]:
# ============================================================
# ???�닝 결과 ?�인 (?�택)
# ============================================================
from IPython.display import Image, display
from pathlib import Path
import yaml, sys
sys.path.insert(0, "src")
from utils import load_config, get_paths

cfg = load_config("config.yaml")
paths = get_paths(cfg)
tune_dir = paths["runs"] / "tune"

# --- 최적 ?�이?�파?��???출력 ---
best_hp_path = tune_dir / "best_hyperparameters.yaml"
print("=" * 52)
print("  ?�� 최적 ?�이?�파?��???(best_hyperparameters.yaml)")
print("=" * 52)
if best_hp_path.exists():
    with open(best_hp_path, "r", encoding="utf-8") as f:
        best_hp = yaml.safe_load(f)
    for k, v in best_hp.items():
        print(f"  {k:<20}: {v}")
    print()
    print(f"?�� ?�일 ?�치: {best_hp_path}")
else:
    print(f"  ???�일 ?�음: {best_hp_path}")
    print("  ?�닝???�료?��? ?�았거나 결과 경로�??�인?�세??")
print("=" * 52)
print()

# --- ?�닝 결과 ?��?지 출력 ---
plots = [
    ("tune_results.png", "?�� ?�닝 ?�렴 결과"),
    ("tune_scatter_plots.png", "?�� ?�라미터 ?�점??"),
]

found = False
for filename, title in plots:
    p = tune_dir / filename
    if p.exists():
        print(title)
        display(Image(str(p)))
        found = True

if not found:
    # Ultralytics 버전???�라 ?�일명이 ?��? ???�으므�??�체 목록 출력
    if tune_dir.exists():
        pngs = list(tune_dir.glob("*.png"))
        if pngs:
            print("?�� tune ?�렉?�리???��?지 ?�일:")
            for p in pngs:
                print(f"  {p.name}")
                display(Image(str(p)))
        else:
            print(f"결과 ?��?지 ?�음: {tune_dir}")
    else:
        print(f"결과 ?�더 ?�음: {tune_dir}")
        print("?�닝???�료?��? ?�았거나 결과 경로�??�인?�세??")

In [ ]:
# Colab ?��????�동 종료 �??�라?�브 ?�기??보장
import time
from google.colab import drive, runtime

print("?�라?�브 ?�일 ?�기?��? ?�해 ?��?�?..")
time.sleep(10) # ?�일 ?�???�료�??�한 ?��?ndrive.flush_and_unmount() # 모델 가중치 ?�이 ?�라?�브???�전???�기?�되?�록 강제 ?�??nprint("?��??�을 �?종료?�니??")
time.sleep(3)
runtime.unassign()